In [1]:
from Strategies.Models.Model_trades import TradesModel, TradesModelRaw, TradesModelCandles
from Math.ti_class import TI_class, TR_class
from Strategies.Backtester_class import BacktesterClass
from Strategies.Strategy_MeanR_NT import Strategy_MeanR_NT
import plotly_express as px
from Strategies.Calibration_class import Calibration
import numpy as np
from Database.TPData import TPData
from Utilities.plotUtils import bokehPlot
import pickle
import pandas as pd
from datetime import  datetime, time

In [15]:
with open(r'X:\Strategies\test\de_fr_m1m2_2023.pkl', 'rb') as pickle_file:
    data_raw = pickle.load(pickle_file)

In [16]:
dem1, frm1, dem2, frm2 = data_raw.values()

In [ ]:
def ret_series(data_series, idx_series):
    df_ret = pd.DataFrame([])
    agg_dict = agg_dict = {'index': 'first', data_series.name: 'mean'}
    data_aux = pd.concat([data_series.reindex(idx_series.index), idx_series],
                         axis=1).reset_index()
    data_aux = data_aux.groupby(0).agg(agg_dict).set_index('index')
    grouped = data_aux.groupby(data_aux.index.date)
    data_dict = {date: group for date, group in grouped}
    for data in data_dict.values():
        ret_aux = np.log(data.fillna(method='ffill')).diff()
        df_ret = pd.concat([df_ret, ret_aux])
    return df_ret.dropna()

agg_dict = {'price': 'mean', 'volume': 'sum'}
data_p = data_raw.set_index('datetime')
start_time = time(8, 0, 0)
end_time = time(18, 0, 0)
data_p = data_p.between_time(start_time, end_time)

# Expected size of candle
data_class = TPData()
data_p = data_class.filter_data(data_p, data_p['price'], 20)
data_p = data_p.groupby(data_p.index).agg(agg_dict)

tau = 85
# EMA
tau_ema = 50
ti_cls = TR_class(tau, tau_ema)
candles = ti_cls.plot_candles(data_p.iloc[:, 0], data_p.iloc[:, 0], volume_series=data_p.iloc[:,1])
candles['p_avg'] = (candles['Open'] + candles['Close'])/2


In [ ]:
from scipy.stats import norm
import matplotlib.pyplot as plt

idx_series = ti_cls.tick_imbalance_indices(data_p.iloc[:, 0])
grouped = idx_series.groupby(idx_series).count()
#grouped.plot(kind='hist', bins=range(1, grouped.max(), int(grouped.max() / 50)))
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

# Calculate distribution of returns
ret = ret_series(data_p.iloc[:, 0], idx_series)
#ret.plot(kind='hist', bins=np.arange(ret.min()[0], ret.max()[0], ret.max()[0] / 50))
plt.figure()
plt.hist(ret, bins=50, density=True, alpha=0.6, color='g')
# Calculate mean and standard deviation
mu, std = ret.mean()[0], ret.std()[0]
skw, kur = ret.skew()[0], ret.kurtosis()[0]
x = np.linspace(ret.min()[0] - std, ret.max()[0] + std, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2)
plt.show()

print('mean: %s, std: %s, skew: %s, curtosis: %s ' % (mu, std, skw, kur))

grouped = idx_series.groupby(idx_series.index.date).agg(['first', 'last'])
grouped = grouped.iloc[:, 1] - grouped.iloc[:, 0] + 1
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

In [ ]:
def run_backTest_w_params(df_input, param_val):
    back_test = BacktesterClass()
    strat = Strategy_MeanR_NT(param_val)
    back_test.simulate_strategy(df_input, strat)
    return strat

In [17]:
follow_dfs = [dem2, frm1, frm2]
lead_df = dem1
b = BacktesterClass()
b.reindex_by_lead(lead_df, follow_dfs)
dem2.reset_index(inplace=True)
frm1.reset_index(inplace=True)
frm2.reset_index(inplace=True)
dem2.set_index('lead_index', inplace=True)
frm1.set_index('lead_index', inplace=True)
frm2.set_index('lead_index', inplace=True)

In [ ]:
follo

In [ ]:
strat = run_backTest_w_params(lead_df, np.array([10, 1, 1, 1.5, 1, 85, 50, 10])) # dopln tau pre emu
# trades_data.set_index('index',inplace=True)
# df_data.set_index('index',inplace=True)
# df_data.reset_index(drop=True,inplace=True)
# df= pd.concat([df_data, trades_data[['pnl','take_profit', 'stop_loss', 'trend']]], axis=1)
# df['returns'] = df['pnl'].fillna(0).cumsum()
# bokehPlot(df, title="strategy sl/tp fixed by position",
#            col_list=[['Close', 'ema', 'take_profit', 'stop_loss', 'openUp', 'openDown', 'emalong'], ['returns']],
#            scatter=['take_profit', 'stop_loss'], sub=2)
# returns.cumsum().plot()
# print('pnl ratio', len(returns[returns > 0])/len(returns))
# print('profit mean/ loss mean', returns[returns > 0].mean(), returns[returns < 0].mean())

In [ ]:
df_data = pd.DataFrame(strat.history_stack)

In [ ]:
a = strat.position_storage
pos = a.export_df()
pos.set_index('open_candle',inplace=True)

In [ ]:
pos['pnl'].cumsum().plot()

In [ ]:
df = pd.concat([df_data, pos[['pnl', 'take_profit', 'stop_loss']]], axis=1)
df['returns'] = df['pnl'].fillna(0).cumsum()
bokehPlot(df, title="strategy sl/tp fixed by position",
           col_list=[['Close', 'ema', 'take_profit', 'stop_loss', 'openUp', 'openDown'], ['returns']],
           scatter=['take_profit', 'stop_loss'], sub=2)

In [ ]:
bokehPlot(df, title="strategy sl/tp fixed by position",
           col_list=[['Close', 'ema', 'openUp', 'openDown']])

In [ ]:
pnls = pd.concat([df_data, trades_data[['pnl', 'trend', 'take_profit', 'stop_loss', 'vol']]], axis=1, join='inner')
trades = pnls.copy()
trades['trend'] = trades['trend'].replace(0, method='bfill')
trades['vol'] = trades['vol'].replace(0, method='bfill')
trades = trades[trades['pnl'] != 0]
trades['intrend'] = trades['vol'] * trades['trend'] > 0
trades['atrend'] = abs(trades['trend'])

In [ ]:
ti_cls = TR_class(tau, tau_ema)
idx_series = ti_cls.tick_imbalance_indices(data_p.iloc[:, 0])
grouped = idx_series.groupby(idx_series).count()
#grouped.plot(kind='hist', bins=range(1, grouped.max(), int(grouped.max() / 50)))
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

# Calculate distribution of returns
ret = ret_series(data_p.iloc[:, 0], idx_series)
#ret.plot(kind='hist', bins=np.arange(ret.min()[0], ret.max()[0], ret.max()[0] / 50))
plt.figure()
plt.hist(ret, bins=50, density=True, alpha=0.6, color='g')
# Calculate mean and standard deviation
mu, std = ret.mean()[0], ret.std()[0]
skw, kur = ret.skew()[0], ret.kurtosis()[0]
x = np.linspace(ret.min()[0] - std, ret.max()[0] + std, 100)
p = norm.pdf(x, mu, std)
plt.plot(x, p, 'k', linewidth=2)
plt.show()

print('mean: %s, std: %s, skew: %s, curtosis: %s ' % (mu, std, skw, kur))

grouped = idx_series.groupby(idx_series.index.date).agg(['first', 'last'])
grouped = grouped.iloc[:, 1] - grouped.iloc[:, 0] + 1
print('mean: %s, median: %s ' % (grouped.mean(), grouped.median()))

In [ ]:
import seaborn as sns
sns.regplot(trades[trades['pnl'] < 6], x='atrend', y='pnl', order=2)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Create the 3D scatter plot
fig = px.scatter_3d(trades[trades['sigma'] < 3], x='abs_trend', y='pnl', z='sigma')

# Add labels and a title
fig.update_layout(scene=dict(xaxis_title='X trend', yaxis_title='Y pnl', zaxis_title='Z sigma'),
                  scene_aspectmode='cube')

# Create a regression plane
X = trades[trades['sigma'] < 1]['trend']
Y = trades[trades['sigma'] < 1]['pnl']
Z = trades[trades['sigma'] < 1]['sigma']

A = np.vstack([X, Z, np.ones(len(X))]).T
model, _, _, _ = np.linalg.lstsq(A, Y, rcond=None)
xx, zz = np.meshgrid(X, Z)
yy = model[0] * xx + model[1] * zz + model[2]

fig.add_trace(go.Surface(x=xx, y=yy, z=zz, opacity=0.8))

# Show the plot
fig.show()

In [ ]:
filtered = trades[trades['abs_trend'] < .03]
print(len(filtered))
filtered['pnl'].cumsum().plot()
print(np.mean(filtered['pnl'])/np.std(filtered['pnl']))
print(np.mean(trades['pnl'])/np.std(trades['pnl']))

In [ ]:
trades_data

In [ ]:
data_class = TPData()

# mkt_list = ['de', 'de', 'de', 'de', 'de', 'de']
# tenor_list = ['m', 'm', 'q', 'q', 'w', 'y']
# tn_list = [1, 2, 1, 2, 1, 1]
# prod = 'base'
# venue_list = ['eex']
# start_date = datetime(2021, 1, 1)
# end_date = datetime(2021, 12, 30)
# n_s = 2

mkt_list = ['de']
tenor_list = ['m']
prod = 'base'
tn1_list = [1]
tn2_list = [2]
venue_list = ['eex']
start_date1 = datetime(2020, 2, 1)
start_date2 = datetime(2020, 3, 1)
end_date = datetime(2023,10,30)
bT = datetime(2020, 1, 1, hour=8, minute=0, second=0)
eT = datetime(2023, 10, 30, hour=18, minute=0, second=0)

dates = pd.date_range(start_date1, end_date, freq='B')
product_date1 = [dates.shift(1, freq='B') if t == 'da' else
                 dates.shift(1, freq='D') if t == 'd' else
                 dates.shift(tn, freq='W-MON') if t == 'w' else
                 (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                 (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                 for t, tn in zip(tenor_list, tn1_list)]
if not tn2_list:
    product_date2 = [None] * len(product_date1)
else:
    product_date2 = [dates.shift(1, freq='B') if t == 'da' else
                     dates.shift(1, freq='D') if t == 'd' else
                     dates.shift(tn, freq='W-MON') if t == 'w' else
                     (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                     (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                     for t, tn in zip(tenor_list, tn2_list)]

if not tn2_list:
    tn_list = [str(t1) for t1 in tn1_list]
else:
    tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]



start_time = time(8, 0, 0)
end_time = time(18, 0, 0)

tr_data_dict = {m + t + n: [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'mean', 'volume': 'sum', 'action': 'first', 'broker_id': 'first'}

for m, t, n, p1_d, p2_d in zip(mkt_list, tenor_list, tn_list, product_date1,
                               product_date2):
    df_tr = pd.DataFrame([])
    if p2_d is None:
        df_prod_dates = pd.DataFrame([p1_d], columns=dates).T
    else:
        df_prod_dates = pd.DataFrame([p1_d, p2_d], columns=dates).T
    for p_d, ds in df_prod_dates.groupby(0).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        if p2_d is None:
            pd_2 = None
        else:
            pd_2 = df_prod_dates.loc[ds[0], 1]
        # Trades
        data_class.create_connection('OracleSQL')
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod, pd_2)
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        #df_tr_aux = data_class.filter_data(df_tr_aux, df_tr_aux['price'], 20)
        df_tr = pd.concat([df_tr, df_tr_aux])
    df_tr = data_class.filter_data(df_tr, df_tr['price'], 20)
    df_tr = df_tr.groupby(df_tr.index).agg(agg_dict)
    tr_data_dict[m + t + str(n)] = df_tr

with open('de_m1m2_2020-23.pkl', 'wb') as f:
    pickle.dump(tr_data_dict, f)